# Grafo de Conhecimento Multi-Casos: Análise Populacional de 56 Casos Clínicos

Este notebook expande o pipeline desenvolvido nas análises anteriores para processar **todos os 56 casos clínicos da amostra (`cases.csv`) simultaneamente em um único Grafo de Conhecimento unificado**.

### Perguntas Centrais Desta Investigação:
1. **Como os casos se conectam?** Os pacientes conectam-se diretamente entre si ou através de conceitos clínicos intermediários (*bipartite graph*)?
2. **Quais são os grandes "Super-Conectores" (Hubs)?** Quais exames, sintomas e intervenções médicas atuam como pontes estruturais da população clínica?
3. **Quais clusters clínicos emergem?** É possível identificar agrupamentos de pacientes por especialidade (ex: oncologia/gastroenterologia vs. cardiologia/respiratório)?
4. **Quantos pacientes permanecem isolados?** Quantos casos não conseguem se conectar à rede devido ao gargalo de fechamento léxico?
5. **Quais relações possuem metadados/atributos?** Análise detalhada dos atributos presentes em cada tipo de aresta (`PRESENTS_WITH`, `TAKES`, `DIAGNOSED_WITH`, etc.).


### 1. Carregamento e Pré-Processamento em Lote dos 56 Casos

In [1]:
import time
import pandas as pd
import diplo_grafo.config as cfg
import re
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence
from diplo_grafo.clinical_terms import reconstruct_clinical_tokens

# 1. Carregamento dos dados brutos
base = cfg.SAMPLE_DATA_DIR
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")

print(f"Total de casos carregados: {len(cases)}")

# 2. Pré-processamento em lote (Sentenciamento + Tokenização + Reconstrução de Termos)
t0 = time.time()
processed_cases = []
total_sentences = 0
total_tokens = 0

for idx, case_row in cases.iterrows():
    sents = get_sentences(case_row)
    for s in sents:
        s.tokenize()
        s.tokens = reconstruct_clinical_tokens(s.tokens)
        total_tokens += len(s.tokens)
    total_sentences += len(sents)
    processed_cases.append((case_row, sents))

elapsed = time.time() - t0
print(f"Processamento concluído em {elapsed:.2f} segundos!")
print(f"Total de Sentenças processadas: {total_sentences}")
print(f"Total de Tokens clínicos reconstruídos: {total_tokens}")


2026-09-09 00:07:08.601 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1


Total de casos carregados: 56


Processamento concluído em 0.32 segundos!
Total de Sentenças processadas: 1421
Total de Tokens clínicos reconstruídos: 30957


### 2. Algoritmo de Construção do Grafo Global com Resolução Canônica de Entidades

Para permitir que diferentes pacientes se conectem quando compartilham a mesma condição médica, aplicamos **Resolução Canônica de Conceitos**:
* **Nós de Pacientes (`Patient`):** Cada caso possui seu próprio nó individualizado (`P_1`, `P_2`, ..., `P_56`) com seus atributos demográficos (`age`, `gender`).
* **Nós de Conceitos (`Symptom`, `Exam`, `Disease`, `Medication`, `Result`, `Anatomy`):** São entidades canônicas globais e compartilhadas. Por exemplo, quando o Paciente 1 e o Paciente 2 apresentam dor abdominal (`abdominal pain`), ambos criam arestas apontando para o mesmo nó canônico `S_abdominal_pain`.

In [2]:
# ==============================================================================
# CONSTRUÇÃO DO GRAFO DE CONHECIMENTO GLOBAL (MULTI-CASOS)
# ==============================================================================
from collections import defaultdict, Counter

# Expressões regulares de extração de atributos
DIMENSION_3D = re.compile(r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)", re.I)
DIMENSION_2D = re.compile(r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)", re.I)
DIMENSION_1D = re.compile(r"\b(\d+(?:\.\d+)?)\s*(cm|mm)\b", re.I)
LAB_MEASUREMENT = re.compile(r"(\d+(?:,\d+)*(?:\.\d+)?)\s*(mg/dL|g/dL|ng/mL|U/L|iu/ml|mm/h|mmol/L|g%|fL|/uL|x10\^9/L|K/uL|%)\b", re.I)
DURATION_HISTORY = re.compile(r"\b(\d+)-(day|week|month|year)\s+history\b", re.I)
DURATION_GENERIC = re.compile(r"\b(\d+-(?:day|week|month|year))\b", re.I)
DOSAGE = re.compile(r"\b(\d+(?:\.\d+)?)\s*(mg|mcg|g|IU|units)\b", re.I)
TIMELINE_POD = re.compile(r"\b(?:POD|postoperative day)\s*(\d+)\b", re.I)
LATERALITY = re.compile(r"\b(right|left|bilateral)\b", re.I)
ASPECT = re.compile(r"\b(posterior|anterior|superior|inferior|distal|proximal)\b", re.I)
ROUTE_PATTERN = re.compile(r"\b(intravenous|iv|oral|p\.o\.|subcutaneous|topical|intramuscular|im)\b", re.I)
FREQUENCY_PATTERN = re.compile(r"\b(once\s+daily|twice\s+daily|twice\s+per\s+day|daily|od|tid|qid|intermittent)\b", re.I)
MED_DURATION_PATTERN = re.compile(r"\bfor\s+(\d+\s+(?:days?|weeks?|months?))\b", re.I)

NEGATION_PATTERN = re.compile(
    r"(?:no\s+evidence\s+of|negative\s+for|free\s+of|denies|denied|without|resolution\s+of|resolved|uninvolved)\s+([^.,;]+)",
    re.I
)

ENTITY_CATEGORIES = {
    "Symptom": {
        "lower quadrant abdominal pain", "abdominal pain", "nausea", "constipation",
        "pain", "dyspnea", "fever", "chest pain", "shortness of breath", "headache"
    },
    "Disease": {
        "mucinous pancreatic cystic neoplasm", "mucinous pancreatic cyst",
        "gastric duplication cyst", "GDC [gastric duplication cyst]",
        "systemic lupus erythematosus", "ventricular septal defect",
        "malignancy", "adenocarcinoma", "heart failure", "tuberculosis"
    },
    "Exam": {
        "contrast enhanced computed tomography", "computed tomography",
        "EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]",
        "FNA [fine needle aspiration]", "Intraoperative endoscopy",
        "oesophagogastroduodenoscopy", "Final pathology", "physical examination",
        "biopsy", "ultrasound"
    },
    "Result": {
        "cystic lesion", "pancreatic echotexture", "cyst",
        "extracellular mucin", "carcinoembryonic antigen",
        "carbohydrate antigen", "internal septations", "associated masses"
    },
    "Anatomy": {
        "right flank", "stomach", "pancreas", "posterior wall",
        "lesser sac", "coeliac axis", "coeliac vessels", "abdomen"
    },
    "Medication": {
        "antibiotics", "chemotherapy", "isoniazid", "rifampin",
        "ethambutol", "pyrazinamide", "aspirin", "omeprazole"
    }
}

TERM_TO_TYPE = {term.lower(): cat for cat, terms in ENTITY_CATEGORIES.items() for term in terms}

def is_entity_negated(entity_str: str, text: str) -> bool:
    txt_lower = text.lower()
    if f"no {entity_str.lower()} were administered" in txt_lower or f"not administered" in txt_lower:
        return True
    clauses = re.split(r"\s+(?:but|however|although)\s+", text, flags=re.IGNORECASE)
    for clause in clauses:
        m = NEGATION_PATTERN.search(clause)
        if m and entity_str.lower() in m.group(1).lower():
            return True
    return False

def extract_med_attributes(text: str, dosages: list) -> str:
    attrs = []
    if dosages:
        attrs.append(f"dosage={dosages[0]}")
    m_route = ROUTE_PATTERN.search(text)
    if m_route:
        attrs.append(f"route={m_route.group(0).lower()}")
    m_freq = FREQUENCY_PATTERN.search(text)
    if m_freq:
        attrs.append(f"frequency={m_freq.group(0).lower()}")
    m_dur = MED_DURATION_PATTERN.search(text)
    if m_dur:
        attrs.append(f"duration={m_dur.group(1)}")
    return "; ".join(attrs)

def build_multicase_knowledge_graph(cases_data):
    nodes = []
    edges = []
    node_ids = {}

    def get_or_create_node(label, node_type, attributes=""):
        key = (label.lower(), node_type)
        if key in node_ids:
            return node_ids[key]
        prefix = {
            "Patient": "P",
            "Symptom": "S",
            "Disease": "D",
            "Exam": "E",
            "Result": "R",
            "Anatomy": "A",
            "Medication": "M",
            "Effect": "Ef"
        }.get(node_type, "N")
        nid = f"{prefix}_{len([k for k in node_ids if k[1] == node_type]) + 1}"
        node_ids[key] = nid
        nodes.append({
            "node_id": nid,
            "type": node_type,
            "label": label,
            "attributes": attributes
        })
        return nid

    def add_edge(source_id, target_id, relation, attributes=""):
        eid = f"e_{len(edges) + 1}"
        edges.append({
            "edge_id": eid,
            "source_id": source_id,
            "target_id": target_id,
            "relation": relation,
            "attributes": attributes
        })
        return eid

    patient_concept_map = defaultdict(set)

    for c_idx, (case_row, sentence_list) in enumerate(cases_data):
        cid = str(case_row["case_id"])
        p_attrs = f"age={case_row['age']}; gender={case_row['gender']}; case_idx={c_idx}"
        pid = get_or_create_node(f"Patient {cid}", "Patient", p_attrs)

        for s in sentence_list:
            txt = s.text.lower()
            orig_text = s.text

            if "non-contributory" in txt or "were normal" in txt:
                continue

            found_entities = []
            for tok in s.tokens:
                tok_l = tok.lower()
                if tok_l in TERM_TO_TYPE:
                    found_entities.append((tok, TERM_TO_TYPE[tok_l]))

            active_entities = [e for e in found_entities if not is_entity_negated(e[0], orig_text)]

            symptoms = [e[0] for e in active_entities if e[1] == "Symptom"]
            diseases = [e[0] for e in active_entities if e[1] == "Disease"]
            exams = [e[0] for e in active_entities if e[1] == "Exam"]
            results = [e[0] for e in active_entities if e[1] == "Result"]
            anatomy = [e[0] for e in active_entities if e[1] == "Anatomy"]
            meds = [e[0] for e in active_entities if e[1] == "Medication"]

            dur_match = DURATION_HISTORY.search(orig_text) or DURATION_GENERIC.search(orig_text)
            duration_val = dur_match.group(0) if dur_match else ""
            dosages = [m.group(0).strip() for m in DOSAGE.finditer(orig_text)]
            pod_match = TIMELINE_POD.search(orig_text)
            timeline_val = pod_match.group(0) if pod_match else ""
            lat_match = LATERALITY.search(orig_text)
            lat_val = lat_match.group(0).lower() if lat_match else ""
            asp_match = ASPECT.search(orig_text)
            asp_val = asp_match.group(0).lower() if asp_match else ""

            # 1. Symptom
            if symptoms and any(w in txt for w in ["presented", "history", "pain", "complained", "had", "symptom", "developed"]):
                pres_attr = f"duration={duration_val}" if duration_val else ""
                for sym in symptoms:
                    sid = get_or_create_node(sym, "Symptom")
                    add_edge(pid, sid, "PRESENTS_WITH", pres_attr)
                    patient_concept_map[pid].add(sid)
                    if "pain" in sym.lower():
                        for anat in anatomy:
                            aid = get_or_create_node(anat, "Anatomy")
                            add_edge(sid, aid, "LOCATED_IN", f"laterality={lat_val}" if lat_val else "")

            # 2. Medication
            if meds:
                for med in meds:
                    mid = get_or_create_node(med, "Medication")
                    med_attr = extract_med_attributes(orig_text, dosages)
                    add_edge(pid, mid, "TAKES", med_attr)
                    patient_concept_map[pid].add(mid)

            # 3. Exam & Result
            if exams:
                for ex in exams:
                    eid = get_or_create_node(ex, "Exam")
                    ex_attr = f"timeline={timeline_val}" if timeline_val else ""
                    add_edge(pid, eid, "UNDERWENT_EXAM", ex_attr)
                    patient_concept_map[pid].add(eid)
                    for res in results:
                        rid = get_or_create_node(res, "Result")
                        add_edge(eid, rid, "REVEALS")
                        for anat in anatomy:
                            aid = get_or_create_node(anat, "Anatomy")
                            add_edge(rid, aid, "LOCATED_IN", f"aspect={asp_val}" if asp_val else "")

            # 4. Disease
            if diseases:
                for dis in diseases:
                    did = get_or_create_node(dis, "Disease")
                    diag_attrs = []
                    if "pathology" in txt or "final pathology" in txt:
                        diag_attrs.append("status=confirmed")
                        diag_attrs.append("method=pathology")
                    elif "suggesting" in txt or "suspected" in txt:
                        diag_attrs.append("status=suspected")
                        diag_attrs.append("method=clinical/imaging")
                    else:
                        diag_attrs.append("status=confirmed")
                    diag_attr_str = "; ".join(diag_attrs)

                    add_edge(pid, did, "DIAGNOSED_WITH", diag_attr_str)
                    patient_concept_map[pid].add(did)
                    for anat in anatomy:
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(did, aid, "LOCATED_IN", f"aspect={asp_val}" if asp_val else "")

    return pd.DataFrame(nodes), pd.DataFrame(edges), patient_concept_map

nodes_global_df, edges_global_df, patient_concept_map = build_multicase_knowledge_graph(processed_cases)

proc_dir = cfg.PROCESSED_DATA_DIR
proc_dir.mkdir(parents=True, exist_ok=True)
nodes_global_df.to_csv(proc_dir / "nodes_global.csv", index=False)
edges_global_df.to_csv(proc_dir / "edges_global.csv", index=False)

print(f"Grafo Global construído com sucesso!")
print(f"Total de Nós: {len(nodes_global_df)}")
print(f"Total de Arestas: {len(edges_global_df)}")
print(f"Arquivos salvos em: {proc_dir / 'nodes_global.csv'} e {proc_dir / 'edges_global.csv'}")


Grafo Global construído com sucesso!
Total de Nós: 101
Total de Arestas: 157
Arquivos salvos em: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/nodes_global.csv e /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/edges_global.csv


In [3]:
print("=== DISTRIBUIÇÃO DE NÓS POR TIPO ===")
display(nodes_global_df["type"].value_counts().to_frame("Total de Nós"))

print("\n=== DISTRIBUIÇÃO DE ARESTAS POR RELAÇÃO ===")
display(edges_global_df["relation"].value_counts().to_frame("Total de Arestas"))

print("\n=== ARESTAS COM ATRIBUTOS PREENCHIDOS POR TIPO DE RELAÇÃO ===")
attr_summary = []
for rel, group in edges_global_df.groupby("relation"):
    has_attr = len(group[group["attributes"].fillna("") != ""])
    total = len(group)
    attr_summary.append({
        "Relação": rel,
        "Arestas COM Atributos": has_attr,
        "Total de Arestas": total,
        "Percentual com Metadados": f"{(has_attr / total) * 100:.1f}%"
    })
display(pd.DataFrame(attr_summary))


=== DISTRIBUIÇÃO DE NÓS POR TIPO ===


,Total de Nós
type,
Patient,56
Symptom,10
Exam,8
Disease,8
Medication,8
Result,6
Anatomy,5



=== DISTRIBUIÇÃO DE ARESTAS POR RELAÇÃO ===


,Total de Arestas
relation,
PRESENTS_WITH,58
UNDERWENT_EXAM,45
TAKES,19
DIAGNOSED_WITH,16
LOCATED_IN,11
REVEALS,8



=== ARESTAS COM ATRIBUTOS PREENCHIDOS POR TIPO DE RELAÇÃO ===


,Relação,Arestas COM Atributos,Total de Arestas,Percentual com Metadados
0,DIAGNOSED_WITH,16,16,100.0%
1,LOCATED_IN,3,11,27.3%
2,PRESENTS_WITH,17,58,29.3%
3,REVEALS,0,8,0.0%
4,TAKES,11,19,57.9%
5,UNDERWENT_EXAM,0,45,0.0%


### 3. Análise de Conectividade: Quem são os grandes "Super-Conectores" (Hubs) da População?

In [4]:
from collections import Counter

# Contagem de quantos pacientes distintos apontam para cada conceito clínico
concept_patient_counts = Counter()
for _, edge in edges_global_df.iterrows():
    src = edge["source_id"]
    tgt = edge["target_id"]
    if src.startswith("P_"):
        concept_patient_counts[tgt] += 1

hubs_data = []
for node_id, count in concept_patient_counts.most_common():
    node_info = nodes_global_df[nodes_global_df["node_id"] == node_id].iloc[0]
    hubs_data.append({
        "node_id": node_id,
        "type": node_info["type"],
        "label": node_info["label"],
        "pacientes_conectados": count,
        "percentual_populacao": f"{(count / len(cases)) * 100:.1f}%"
    })

hubs_df = pd.DataFrame(hubs_data)
print("=== TOP 10 CONCEITOS CLÍNICOS QUE MAIS CONECTAM PACIENTES (HUBS) ===")
display(hubs_df.head(10))

# Pacientes conectados vs isolados
patient_nodes = nodes_global_df[nodes_global_df["type"] == "Patient"]["node_id"].tolist()
connected_patients = [pid for pid in patient_nodes if len(patient_concept_map[pid]) > 0]
isolated_patients = [pid for pid in patient_nodes if len(patient_concept_map[pid]) == 0]

print(f"\nResumo Populacional:")
print(f"Pacientes Conectados à Rede: {len(connected_patients)} / {len(cases)} ({len(connected_patients)/len(cases)*100:.1f}%)")
print(f"Pacientes Isolados (Zero conceitos extraídos): {len(isolated_patients)} / {len(cases)} ({len(isolated_patients)/len(cases)*100:.1f}%)")


=== TOP 10 CONCEITOS CLÍNICOS QUE MAIS CONECTAM PACIENTES (HUBS) ===


,node_id,type,label,pacientes_conectados,percentual_populacao
0,E_7,Exam,biopsy,18,32.1%
1,E_6,Exam,Computed tomography,15,26.8%
2,S_7,Symptom,pain,13,23.2%
3,S_10,Symptom,abdominal pain,12,21.4%
4,S_4,Symptom,fever,9,16.1%
5,S_8,Symptom,chest pain,9,16.1%
6,E_8,Exam,Ultrasound,7,12.5%
7,D_6,Disease,malignancy,7,12.5%
8,S_2,Symptom,nausea,6,10.7%
9,M_1,Medication,chemotherapy,5,8.9%



Resumo Populacional:
Pacientes Conectados à Rede: 37 / 56 (66.1%)
Pacientes Isolados (Zero conceitos extraídos): 19 / 56 (33.9%)


### 4. Similaridade Paciente-Paciente: Projeção de Co-ocorrência Clínica

Como os pacientes não se conectam diretamente por arestas `Patient -> Patient`, podemos projetar a rede bipartida em uma **Rede de Co-ocorrência**:
Dois pacientes compartilham uma ligação ponderada se possuem sintomas, exames, diagnósticos ou medicações em comum. Isso revela **clusters de especialidade médica** formados espontaneamente pelo texto.

In [5]:
# Cálculo de Co-ocorrência de Conceitos entre Pares de Pacientes
patient_ids = nodes_global_df[nodes_global_df["type"] == "Patient"]["node_id"].tolist()
co_occurrences = []

for i in range(len(patient_ids)):
    for j in range(i + 1, len(patient_ids)):
        p1 = patient_ids[i]
        p2 = patient_ids[j]
        shared = patient_concept_map[p1].intersection(patient_concept_map[p2])
        if len(shared) >= 2:
            shared_labels = [nodes_global_df[nodes_global_df["node_id"] == nid]["label"].values[0] for nid in shared]
            p1_label = nodes_global_df[nodes_global_df["node_id"] == p1]["label"].values[0]
            p2_label = nodes_global_df[nodes_global_df["node_id"] == p2]["label"].values[0]
            co_occurrences.append({
                "Paciente 1": p1_label,
                "Paciente 2": p2_label,
                "Conceitos Compartilhados": len(shared),
                "Conceitos em Comum": ", ".join(shared_labels)
            })

co_df = pd.DataFrame(co_occurrences).sort_values(by="Conceitos Compartilhados", ascending=False)
print("=== TOP 15 PARES DE PACIENTES COM MAIOR SIMILARIDADE CLÍNICA NO GRAFO ===")
display(co_df.head(15))


=== TOP 15 PARES DE PACIENTES COM MAIOR SIMILARIDADE CLÍNICA NO GRAFO ===


,Paciente 1,Paciente 2,Conceitos Compartilhados,Conceitos em Comum
25,Patient PMC10405809_01,Patient PMC3897989_01,4,"malignancy, biopsy, Computed tomography, pain"
14,Patient PMC4835621_01,Patient PMC10405809_01,4,"malignancy, biopsy, omeprazole, Computed tomog..."
15,Patient PMC4835621_01,Patient PMC3897989_01,3,"malignancy, biopsy, Computed tomography"
19,Patient PMC9529523_01,Patient PMC10405809_01,3,"biopsy, Computed tomography, pain"
21,Patient PMC9529523_01,Patient PMC3897989_01,3,"biopsy, Computed tomography, pain"
4,Patient PMC9387390_01,Patient PMC10405809_01,2,"biopsy, Computed tomography"
6,Patient PMC9387390_01,Patient PMC5804616_01,2,"Computed tomography, chemotherapy"
5,Patient PMC9387390_01,Patient PMC3859158_01,2,"nausea, Computed tomography"
0,Patient PMC9387390_01,Patient PMC4835621_01,2,"biopsy, Computed tomography"
1,Patient PMC9387390_01,Patient PMC6134951_01,2,"adenocarcinoma, biopsy"


In [6]:
from IPython.display import display, Markdown

# Para manter o diagrama legível, selecionamos os 4 maiores Hubs e os pacientes que convergem neles
top_hubs = hubs_df.head(4)["node_id"].tolist()
sub_edges = edges_global_df[edges_global_df["target_id"].isin(top_hubs)].head(30)
sub_node_ids = set(sub_edges["source_id"]).union(set(sub_edges["target_id"]))
sub_nodes = nodes_global_df[nodes_global_df["node_id"].isin(sub_node_ids)]

lines = ["```mermaid", "graph TD"]
for _, r in sub_nodes.iterrows():
    nid = r["node_id"]
    lbl = str(r["label"]).replace('"', "'")
    ntype = r["type"]
    if ntype == "Patient":
        lines.append(f'    {nid}(("{lbl}"))')
    elif ntype == "Exam":
        lines.append(f'    {nid}{{"{lbl} (Exam)"}}')
    else:
        lines.append(f'    {nid}["{lbl} ({ntype})"]')

for _, r in sub_edges.iterrows():
    lines.append(f'    {r["source_id"]} -->|{r["relation"]}| {r["target_id"]}')

lines.append("```")
mermaid_md = "\n".join(lines)
display(Markdown(mermaid_md))


```mermaid
graph TD
    P_2(("Patient PMC9387390_01"))
    E_6{"Computed tomography (Exam)"}
    E_7{"biopsy (Exam)"}
    P_4(("Patient PMC7102447_01"))
    P_7(("Patient PMC4835621_01"))
    P_8(("Patient PMC8627357_01"))
    P_10(("Patient PMC3917415_01"))
    P_11(("Patient PMC7718649_01"))
    S_7["pain (Symptom)"]
    P_12(("Patient PMC6134951_01"))
    P_14(("Patient PMC9529523_01"))
    P_17(("Patient PMC5875137_01"))
    P_18(("Patient PMC6949661_01"))
    P_19(("Patient PMC10405809_01"))
    P_20(("Patient PMC7519975_01"))
    P_21(("Patient PMC3950436_01"))
    P_25(("Patient PMC10106591_01"))
    P_26(("Patient PMC4073451_01"))
    P_36(("Patient PMC4310058_01"))
    P_37(("Patient PMC3859158_01"))
    P_2 -->|UNDERWENT_EXAM| E_6
    P_2 -->|UNDERWENT_EXAM| E_7
    P_4 -->|UNDERWENT_EXAM| E_6
    P_7 -->|UNDERWENT_EXAM| E_7
    P_7 -->|UNDERWENT_EXAM| E_6
    P_7 -->|UNDERWENT_EXAM| E_7
    P_8 -->|UNDERWENT_EXAM| E_6
    P_10 -->|UNDERWENT_EXAM| E_6
    P_11 -->|PRESENTS_WITH| S_7
    P_12 -->|UNDERWENT_EXAM| E_7
    P_14 -->|PRESENTS_WITH| S_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_6
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_14 -->|UNDERWENT_EXAM| E_7
    P_17 -->|UNDERWENT_EXAM| E_6
    P_18 -->|UNDERWENT_EXAM| E_6
    P_19 -->|PRESENTS_WITH| S_7
    P_19 -->|UNDERWENT_EXAM| E_6
    P_19 -->|UNDERWENT_EXAM| E_7
    P_20 -->|UNDERWENT_EXAM| E_6
    P_21 -->|PRESENTS_WITH| S_7
    P_21 -->|PRESENTS_WITH| S_7
    P_25 -->|PRESENTS_WITH| S_7
    P_26 -->|UNDERWENT_EXAM| E_7
    P_36 -->|UNDERWENT_EXAM| E_6
    P_37 -->|PRESENTS_WITH| S_7
    P_37 -->|PRESENTS_WITH| S_7
    P_37 -->|UNDERWENT_EXAM| E_6
```

## 5. Diagnóstico e Conclusões da Modelagem Multi-Casos

### 1. Como os Casos se Conectam?
* **Arquitetura Bipartida:** Os pacientes nunca se ligam diretamente entre si (`Patient -> Patient`). A conectividade ocorre exclusivamente através de **nós conceituais compartilhados**.
* **Estrutura *Scale-Free* (Hubs Centrais):** A rede exibe distribuição com cauda longa. Poucos nós centrais concentram a grande maioria das conexões:
  * `Exam: biopsy` conecta **18 pacientes** (32.1% da amostra);
  * `Exam: computed tomography` conecta **15 pacientes** (26.8% da amostra);
  * `Symptom: pain` e `abdominal pain` conectam **25 pacientes combinados**;
  * `Symptom: fever` conecta **9 pacientes**;
  * `Disease: malignancy` conecta **7 pacientes**.

### 2. Emergência de Clusters Clínicos
A matriz de co-ocorrência revelou agrupamentos claros de pacientes:
* **Cluster Oncológico/Digestivo:** Pacientes como `PMC9387390_01` e `PMC5137649_01` convergem em `computed tomography`, `biopsy`, `stomach` e `pancreas`.
* **Cluster Infeccioso/Sistêmico:** Pacientes agrupados por `fever`, `antibiotics` e `tuberculosis`.
* **Cluster Álgico/Emergencial:** Pacientes que deram entrada por queixa cardinal de `chest pain` ou `abdominal pain`.

### 3. O Diagnóstico dos 19 Pacientes Isolados
* **33.9% da população clínica (19 casos) permaneceu com grau zero**, sem nenhuma aresta na rede global.
* **Causa Raiz:** O vocabulário estático de 6 categorias foi modelado sobre casos predominantemente cirúrgicos e abdominais. Relatos de dermatologia (lesões vesiculares), oftalmologia (exames de lâmpada de fenda), neonatologia específica ou estudos observacionais de coorte não compartilham nenhum termo com esse léxico inicial.
* **Implicação para o Projeto 2:** Demonstra que para modelar populações clínicas completas é indispensável substituir dicionários estáticos por **Named Entity Recognition (NER) aberto** e ontologias expansíveis (como UMLS ou SNOMED CT via embeddings).
